In [1]:
!apt-get install openjdk-11-jdk -y
!pip install pyspark

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  at-spi2-core fonts-dejavu-core fonts-dejavu-extra gsettings-desktop-schemas
  libatk-bridge2.0-0 libatk-wrapper-java libatk-wrapper-java-jni libatk1.0-0
  libatk1.0-data libatspi2.0-0 libxcomposite1 libxt-dev libxtst6 libxxf86dga1
  openjdk-11-jdk-headless openjdk-11-jre openjdk-11-jre-headless
  session-migration x11-utils
Suggested packages:
  libxt-doc openjdk-11-demo openjdk-11-source visualvm libnss-mdns
  fonts-ipafont-gothic fonts-ipafont-mincho fonts-wqy-microhei
  | fonts-wqy-zenhei fonts-indic mesa-utils
The following NEW packages will be installed:
  at-spi2-core fonts-dejavu-core fonts-dejavu-extra gsettings-desktop-schemas
  libatk-bridge2.0-0 libatk-wrapper-java libatk-wrapper-java-jni libatk1.0-0
  libatk1.0-data libatspi2.0-0 libxcomposite1 libxt-dev libxtst6 libxxf86dga1
  openjdk-11-jdk openjdk-11-jdk-headless openjdk-

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Configuration
DATA_PATH = "/content/drive/My Drive/ProjectBigData/01 datasets/hospital_prices_clean"
OUTPUT_PATH = "/content/drive/My Drive/ProjectBigData/05 artifacts/model_v2"
MODEL_PATH = f"{OUTPUT_PATH}/hospital_price_pipeline"


In [4]:
import os
os.makedirs(OUTPUT_PATH, exist_ok=True)

# Standard imports
import numpy as np
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType
import warnings
warnings.filterwarnings('ignore')

print("Imports loaded")

Imports loaded


In [ ]:
import os
from pyspark.sql import SparkSession
from google.colab import drive
import re
from pyspark.sql import functions as F

In [5]:
spark = (SparkSession.builder
         .appName("HospitalChargesModelV2")
         .config('spark.driver.memory', "8g")
         .config("spark.executor.memory", "8g")
         .getOrCreate())

spark

In [8]:
# Reading the data
hospital_data = spark.read.parquet(DATA_PATH)
print(f"Loaded {hospital_data.count():,} rows")
print("\nSchema:")
hospital_data.printSchema()

print("\nFirst 5 rows:")
hospital_data.show(5, truncate=False)

Loaded 47,224 rows

Schema:
root
 |-- hospital_name: string (nullable = true)
 |-- code_2: string (nullable = true)
 |-- setting: string (nullable = true)
 |-- discounted_cash: double (nullable = true)
 |-- gross_price: double (nullable = true)
 |-- description: string (nullable = true)
 |-- code_1: string (nullable = true)
 |-- code_1_type: string (nullable = true)
 |-- code_2_type: string (nullable = true)
 |-- code_3: string (nullable = true)


First 5 rows:
+-------------+------+----------+---------------+-----------+---------------------------------------------+-----------+-----------+-----------+------+
|hospital_name|code_2|setting   |discounted_cash|gross_price|description                                  |code_1     |code_1_type|code_2_type|code_3|
+-------------+------+----------+---------------+-----------+---------------------------------------------+-----------+-----------+-----------+------+
|massachusetts|0001U |outpatient|987.75         |1317.0     |rbc dna hea 35 ag 11

### Data Preparation for Modelling

In [10]:
# 1. Remove invalid target/gross rows
df_clean = hospital_data.filter(
    (F.col("discounted_cash").isNotNull()) &
    (F.col("discounted_cash") > 0) &
    (F.col("gross_price").isNotNull()) &
    (F.col("gross_price") > 0)
)

print(f"Rows after filtering invalid prices: {df_clean.count():,}")


Rows after filtering invalid prices: 47,224


In [11]:
# 2. Replace missing textual and categorical fields
df_clean = df_clean.fillna({
    "description": "",
    "code_2": "NO_CODE_2",
    "code_2_type": "unknown",
    "code_1": "NONE",
    "code_1_type": "unknown",
    "code_3": "NO_CODE_3",
    "setting": "unknown_setting",
    "hospital_name": "unknown_hospital"
})

# Verify no nulls remain
print("\nNull counts after filling:")
nulls_found = False
for c in df_clean.columns:
    null_count = df_clean.filter(F.col(c).isNull()).count()
    if null_count > 0:
        print(f"  {c}: {null_count}")
        nulls_found = True
if not nulls_found:
    print("No null values found")


Null counts after filling:
No null values found


In [12]:
# 3. Log-transform target and numeric predictor
df_model = (
    df_clean
    .withColumn("log_cash", F.log(F.col("discounted_cash")))
    .withColumn("log_gross", F.log(F.col("gross_price")))
)

print("Log transformation complete")
print(f"\nSample log-transformed values:")
df_model.select("discounted_cash", "log_cash", "gross_price", "log_gross").show(5)

Log transformation complete

Sample log-transformed values:
+---------------+-----------------+-----------+-----------------+
|discounted_cash|         log_cash|gross_price|        log_gross|
+---------------+-----------------+-----------+-----------------+
|         987.75|  6.8954296292915|     1317.0|7.183111701743281|
|          381.0|5.942799375126701|      508.0|6.230481447578482|
|        1301.25|7.171080619929179|     1735.0| 7.45876269238096|
|         6417.0|8.766705997750515|     8556.0|9.054388070202297|
|         6417.0|8.766705997750515|     8556.0|9.054388070202297|
+---------------+-----------------+-----------+-----------------+
only showing top 5 rows



## Train/Validation/Test Split

In [13]:
# Split data: 70% train, 15% validation, 15% test
train_df, val_df, test_df = df_model.randomSplit([0.7, 0.15, 0.15], seed=42)

train_count = train_df.count()
val_count = val_df.count()
test_count = test_df.count()
total_count = train_count + val_count + test_count

print(f"Train set:      {train_count:,} rows ({train_count/total_count*100:.1f}%)")
print(f"Validation set: {val_count:,} rows ({val_count/total_count*100:.1f}%)")
print(f"Test set:       {test_count:,} rows ({test_count/total_count*100:.1f}%)")
print(f"Total:          {total_count:,} rows")


Train set:      33,221 rows (70.3%)
Validation set: 6,980 rows (14.8%)
Test set:       7,023 rows (14.9%)
Total:          47,224 rows


In [ ]:
# # Save splits
# splits_path = f"{OUTPUT_PATH}/data_splits"
# os.makedirs(splits_path, exist_ok=True)

# train_df.write.mode("overwrite").parquet(f"{splits_path}/train")
# val_df.write.mode("overwrite").parquet(f"{splits_path}/validation")
# test_df.write.mode("overwrite").parquet(f"{splits_path}/test")

# print(f"Data splits saved to {splits_path}")

## Build Pipeline

In [20]:
from pyspark.ml.feature import (
    Tokenizer, StopWordsRemover, HashingTF, IDF,
    StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
)
from pyspark.ml.regression import GBTRegressor  # Gradient Boosted Trees
# Alternative imports: RandomForestRegressor, DecisionTreeRegressor, LinearRegression
from pyspark.ml import Pipeline

# Define feature columns
text_col = "description"

categorical_cols = [
    "hospital_name",
    "code_2",
    "setting",
    "code_1_type",
    "code_2_type"
]

numeric_cols = ["log_gross"]
target_col = "log_cash"

print("Feature columns defined")
print(f"  Text: {text_col}")
print(f"  Categorical: {categorical_cols}")
print(f"  Numeric: {numeric_cols}")
print(f"  Target: {target_col}")


Feature columns defined
  Text: description
  Categorical: ['hospital_name', 'code_2', 'setting', 'code_1_type', 'code_2_type']
  Numeric: ['log_gross']
  Target: log_cash


In [16]:
# Text preprocessing: TF-IDF
tokenizer = Tokenizer(inputCol=text_col, outputCol="words")
remover = StopWordsRemover(inputCol="words", outputCol="filtered")
hashing_tf = HashingTF(inputCol="filtered", outputCol="tf_raw", numFeatures=20000)
idf = IDF(inputCol="tf_raw", outputCol="tfidf_features")

print("Text preprocessing stages defined")

Text preprocessing stages defined


In [17]:
# Categorical variables: StringIndexer + OneHotEncoder
indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    for c in categorical_cols
]

encoders = [
    OneHotEncoder(inputCols=[f"{c}_idx"], outputCols=[f"{c}_vec"])
    for c in categorical_cols
]

print(f"Categorical encoding stages defined ({len(indexers)} indexers, {len(encoders)} encoders)")

Categorical encoding stages defined (5 indexers, 5 encoders)


In [18]:
# Numeric variables: StandardScaler
num_assembler = VectorAssembler(
    inputCols=numeric_cols,
    outputCol="num_unscaled"
)

scaler = StandardScaler(
    inputCol="num_unscaled",
    outputCol="num_scaled",
    withMean=True,
    withStd=True
)

print("Numeric scaling stages defined")

Numeric scaling stages defined


In [19]:
# Assembling final feature vector
feature_cols = ["tfidf_features"] + [f"{c}_vec" for c in categorical_cols] + ["num_scaled"]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

print(f"Feature assembler defined")
print(f"Feature columns: {len(feature_cols)} components")

Feature assembler defined
Feature columns: 7 components


In [ ]:
# # Define Linear Regression model
# lr = LinearRegression(
#     featuresCol="features",
#     labelCol=target_col,
#     maxIter=50,
#     regParam=0.1,
#     elasticNetParam=0.0,
#     solver="l-bfgs"  # Better for large datasets
# )

# print("Linear Regression model defined")

In [21]:
gbt = GBTRegressor(
    featuresCol="features",
    labelCol=target_col,
    maxDepth=5,           # Maximum depth of trees
    maxIter=50,           # Number of trees in the ensemble
    stepSize=0.1,         # Learning rate (shrinkage)
    subsamplingRate=0.8,  # Fraction of training data used per tree
    minInstancesPerNode=10,  # Minimum instances per leaf
    maxBins=32,           # Number of bins for discretizing continuous features
    seed=42               # Random seed for reproducibility
)

print("Gradient Boosted Trees (GBT) Regressor model defined")
print("  - Ensemble of 50 trees")
print("  - Max depth: 5")
print("  - Learning rate: 0.1")


Gradient Boosted Trees (GBT) Regressor model defined
  - Ensemble of 50 trees
  - Max depth: 5
  - Learning rate: 0.1


In [22]:
# Build complete pipeline
pipeline = Pipeline(stages=
    [tokenizer, remover, hashing_tf, idf] +
    indexers + encoders +
    [num_assembler, scaler, assembler, gbt]
)

print("Complete ML pipeline built")
print(f"  Total stages: {len(pipeline.getStages())}")
print(f"  Model: Gradient Boosted Trees (GBT) Regressor")

Complete ML pipeline built
  Total stages: 18
  Model: Gradient Boosted Trees (GBT) Regressor


## Train Model

In [ ]:
## Train Model

In [23]:
print("-" * 80)
print("TRAINING MODEL")
print("-" * 80)
print("\nFitting preprocessing pipeline...")

# Fit pipeline on training data
pipeline_model = pipeline.fit(train_df)

print("Pipeline fitted successfully")

--------------------------------------------------------------------------------
TRAINING MODEL
--------------------------------------------------------------------------------

Fitting preprocessing pipeline...
Pipeline fitted successfully


In [24]:
print("\nGenerating predictions...")

# Transform all splits
train_pred = pipeline_model.transform(train_df)
val_pred = pipeline_model.transform(val_df)
test_pred = pipeline_model.transform(test_df)

print("Predictions generated for all splits")


Generating predictions...
Predictions generated for all splits


## Evaluate Model

In [25]:
from pyspark.ml.evaluation import RegressionEvaluator

def evaluate(df, label="log_cash"):
    """Evaluate model performance"""
    evaluator = RegressionEvaluator(labelCol=label, predictionCol="prediction")

    rmse = evaluator.setMetricName("rmse").evaluate(df)
    mae = evaluator.setMetricName("mae").evaluate(df)
    r2 = evaluator.setMetricName("r2").evaluate(df)

    # Calculate MAPE manually (on original scale, not log scale)
    df_m = df.withColumn("ape", F.abs((F.exp(F.col("prediction")) - F.exp(F.col(label))) / F.exp(F.col(label))))
    mape = df_m.agg(F.mean("ape")).first()[0]

    return {"RMSE": rmse, "MAE": mae, "R2": r2, "MAPE": mape}

print("Evaluating model performance...")
print("\n" + "=" * 80)

train_metrics = evaluate(train_pred)
print("TRAIN Metrics:")
for metric, value in train_metrics.items():
    if metric == "MAPE":
        print(f"  {metric}: {value*100:.2f}%")
    elif metric == "R2":
        print(f"  {metric}: {value:.4f}")
    else:
        print(f"  {metric}: {value:.4f}")

print("\n" + "-" * 80)

val_metrics = evaluate(val_pred)
print("VALIDATION Metrics:")
for metric, value in val_metrics.items():
    if metric == "MAPE":
        print(f"  {metric}: {value*100:.2f}%")
    elif metric == "R2":
        print(f"  {metric}: {value:.4f}")
    else:
        print(f"  {metric}: {value:.4f}")

print("\n" + "-" * 80)

test_metrics = evaluate(test_pred)
print("TEST Metrics:")
for metric, value in test_metrics.items():
    if metric == "MAPE":
        print(f"  {metric}: {value*100:.2f}%")
    elif metric == "R2":
        print(f"  {metric}: {value:.4f}")
    else:
        print(f"  {metric}: {value:.4f}")


Evaluating model performance...

TRAIN Metrics:
  RMSE: 0.2406
  MAE: 0.1140
  R2: 0.9868
  MAPE: 17.66%

--------------------------------------------------------------------------------
VALIDATION Metrics:
  RMSE: 0.2786
  MAE: 0.1195
  R2: 0.9822
  MAPE: 30.84%

--------------------------------------------------------------------------------
TEST Metrics:
  RMSE: 0.2309
  MAE: 0.1151
  R2: 0.9877
  MAPE: 19.17%


## Save Model

In [26]:
# Save the trained pipeline model
print(f"Saving model to {MODEL_PATH}...")
pipeline_model.write().overwrite().save(MODEL_PATH)
print(f"Model saved successfully to {MODEL_PATH}")

Saving model to /content/drive/My Drive/ProjectBigData/05 artifacts/model_v2/hospital_price_pipeline...
Model saved successfully to /content/drive/My Drive/ProjectBigData/05 artifacts/model_v2/hospital_price_pipeline


In [ ]:
## Linear Regression results
# print("TRAIN:", evaluate(train_pred))
# print("VALID:", evaluate(val_pred))
# print("TEST:", evaluate(test_pred))


TRAIN: {'RMSE': 0.17508234667697603, 'MAE': 0.0823278982612729, 'R2': 0.9930655054521644, 'MAPE': 0.0968552143378268}
VALID: {'RMSE': 0.2811988415004916, 'MAE': 0.12550898219485113, 'R2': 0.9813201149550008, 'MAPE': 0.24203867231511486}
TEST: {'RMSE': 0.2732444927036042, 'MAE': 0.12514627550987256, 'R2': 0.982796110234425, 'MAPE': 0.23323983126728431}


In [27]:
import math

# Example: Predict price for MRI procedure
test_row = [{
    "description": "mri lower extremity joint w contrast",
    "hospital_name": "massachusetts",
    "code_2": "73723",
    "code_2_type": "cpt",
    "code_1": "PX-31000022",
    "code_1_type": "cdm",
    "code_3": "0352",
    "setting": "outpatient",
    "gross_price": 4530.00,
    "log_gross": math.log(4530.00)
}]

user_input_df = spark.createDataFrame(test_row)

# Make prediction
pred = pipeline_model.transform(user_input_df)

# Transform back to original scale
pred = pred.withColumn("pred_cash", F.exp("prediction"))

print("Prediction Results:")
pred.select("hospital_name", "description", "gross_price", "pred_cash").show(truncate=False)


Prediction Results:
+-------------+------------------------------------+-----------+------------------+
|hospital_name|description                         |gross_price|pred_cash         |
+-------------+------------------------------------+-----------+------------------+
|massachusetts|mri lower extremity joint w contrast|4530.0     |3243.6896953450623|
+-------------+------------------------------------+-----------+------------------+



In [28]:
# Look up actual target value from the dataset for comparison
# Use the same test parameters to find matching records

test_params = {
    "hospital_name": "massachusetts",
    "code_2": "73723",
    "setting": "outpatient",
    "description": "mri lower extremity joint w contrast"
}

# Filter dataset to find matching records
actual_df = df_model.filter(
    (F.col("hospital_name") == test_params["hospital_name"]) &
    (F.col("code_2") == test_params["code_2"]) &
    (F.col("setting") == test_params["setting"]) &
    (F.lower(F.col("description")).contains(test_params["description"].lower()))
).select(
    "hospital_name",
    "description",
    "gross_price",
    "discounted_cash",
    "log_cash"
)

print("=" * 80)
print("ACTUAL TARGET VALUES FROM DATASET")
print("=" * 80)

if actual_df.count() > 0:
    print(f"\nFound {actual_df.count()} matching record(s):\n")
    actual_df.show(truncate=False)

    # Get the actual values for comparison
    actual_values = actual_df.select(
        F.avg("discounted_cash").alias("avg_discounted_cash"),
        F.avg("log_cash").alias("avg_log_cash"),
        F.percentile_approx("discounted_cash", 0.5).alias("median_discounted_cash")
    ).collect()[0]

    # Get predicted value
    pred_value = pred.select("pred_cash").collect()[0][0]
    pred_log_value = pred.select("prediction").collect()[0][0]

    print("\n" + "=" * 80)
    print("PREDICTED vs ACTUAL COMPARISON")
    print("=" * 80)
    print(f"\nPredicted Discounted Cash Price: ${pred_value:,.2f}")
    print(f"Predicted (log scale): {pred_log_value:.4f}")
    print(f"\nActual Average Discounted Cash Price: ${actual_values['avg_discounted_cash']:,.2f}")
    print(f"Actual Median Discounted Cash Price: ${actual_values['median_discounted_cash']:,.2f}")
    print(f"Actual Average (log scale): {actual_values['avg_log_cash']:.4f}")

    # Calculate differences
    diff = pred_value - actual_values['avg_discounted_cash']
    diff_pct = (diff / actual_values['avg_discounted_cash']) * 100 if actual_values['avg_discounted_cash'] > 0 else 0

    print(f"\n{'=' * 80}")
    print("DIFFERENCE ANALYSIS")
    print(f"{'=' * 80}")
    print(f"Absolute Difference: ${abs(diff):,.2f}")
    print(f"Percentage Error: {abs(diff_pct):.2f}%")

    if abs(diff_pct) < 5:
        print(f" Excellent prediction (within 5%)")
    elif abs(diff_pct) < 10:
        print(f" Good prediction (within 10%)")
    elif abs(diff_pct) < 20:
        print(f"Acceptable prediction (within 20%)")
    else:
        print(f"Large deviation (>20%)")

else:
    print("\nNo matching records found in dataset with those exact parameters.")
    print("Trying fuzzy match on description...")

    # Try fuzzy match
    fuzzy_df = df_model.filter(
        (F.col("hospital_name").contains(test_params["hospital_name"].lower())) &
        (F.col("code_2") == test_params["code_2"]) &
        (F.col("setting") == test_params["setting"])
    ).select(
        "hospital_name",
        "description",
        "gross_price",
        "discounted_cash"
    ).limit(5)

    if fuzzy_df.count() > 0:
        print(f"\nFound {fuzzy_df.count()} similar record(s):\n")
        fuzzy_df.show(truncate=False)
    else:
        print("\nNo similar records found.")


ACTUAL TARGET VALUES FROM DATASET

No matching records found in dataset with those exact parameters.
Trying fuzzy match on description...

Found 1 similar record(s):

+-------------+----------------------------+-----------+---------------+
|hospital_name|description                 |gross_price|discounted_cash|
+-------------+----------------------------+-----------+---------------+
|massachusetts|mri lwr ext joint w&w/o cont|4372.0     |3279.0         |
+-------------+----------------------------+-----------+---------------+



## Tetsing on one data point

In [ ]:
# # Loading parquet files

# from pyspark.ml.pipeline import PipelineModel

# loaded_pipeline = PipelineModel.load("/content/drive/My Drive/ProjectBigData/05 artifacts/hospital_price_pipeline")


In [ ]:
# matched_description = "mri lower extremity joint w contrast"
# code2 = "73723"              # CPT for MRI knee with contrast
# code2_type = "cpt"
# code1_type = "cdm"
# code1 = "PX-31000022"
# code3 = "0352"
# setting = "outpatient"
# hospital = "massachusetts"

In [ ]:
# import math

# test_row = [{
#     "description": "mri lower extremity joint w contrast",
#     "hospital_name": "massachusetts",
#     "code_2": "73723",
#     "code_2_type": "cpt",
#     "code_1": "PX-31000022",
#     "code_1_type": "cdm",
#     "code_3": "0352",
#     "setting": "outpatient",
#     "gross_price": 4530.00,
#     "log_gross": math.log(4530.00)
# }]

# user_input_df = spark.createDataFrame(test_row)


In [ ]:
# pred = loaded_pipeline.transform(user_input_df)
# pred.select("prediction").show()
# pred.withColumn("pred_cash", F.exp("prediction")).select("pred_cash").show()

+-----------------+
|       prediction|
+-----------------+
|7.752202004535626|
+-----------------+

+----------------+
|       pred_cash|
+----------------+
|2326.69016017937|
+----------------+



In [ ]:
# # Testing all hospitals at once

# hospitals = [
#     ("massachusetts", 4530.00),
#     ("newton-wellesley", 3825.00),
#     ("salem", 3990.00),
#     ("nantucket cottage", 2990.00),
#     ("mclean", 1850.00)
# ]

# input_rows = []

# for hospital, gp in hospitals:
#     input_rows.append({
#         "description": "mri lower extremity joint w contrast",
#         "hospital_name": hospital,
#         "code_2": "73723",
#         "code_2_type": "cpt",
#         "code_1": "PX-31000022",
#         "code_1_type": "cdm",
#         "code_3": "0352",
#         "setting": "outpatient",
#         "gross_price": gp,
#         "log_gross": math.log(gp)
#     })

# user_input_df = spark.createDataFrame(input_rows)

# pred = loaded_pipeline.transform(user_input_df)
# pred = pred.withColumn("pred_cash", F.exp("prediction"))

# pred.select("hospital_name", "pred_cash").show()


+-----------------+------------------+
|    hospital_name|         pred_cash|
+-----------------+------------------+
|    massachusetts|  2326.69016017937|
| newton-wellesley| 1955.553579039184|
|            salem|2004.8728019213079|
|nantucket cottage|1635.3069896057073|
|           mclean| 928.7064137252742|
+-----------------+------------------+

